In [21]:
from src.utils import load_config
from src.models.personalized_ad_frequency import MODEL_REGISTRY
import pandas as pd
import os
from sklearn.preprocessing import StandardScaler

config = load_config("configs/personalized_ad_frequency.yaml")


In [22]:
# Veri seti yoksa hata logu basar.
processed_data_path = config['processed_data_path']
if not os.path.exists(processed_data_path):
    print(f"❌ HATA: Gerekli bir dosya bulunamadı -> {processed_data_path}")
        

In [23]:
df = pd.read_excel(processed_data_path)

In [24]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3923 entries, 0 to 3922
Data columns (total 11 columns):
 #   Column                             Non-Null Count  Dtype  
---  ------                             --------------  -----  
 0   user_key_id                        3923 non-null   object 
 1   country                            3923 non-null   object 
 2   age                                3923 non-null   int64  
 3   sex                                3923 non-null   object 
 4   easy_completion_rate               3923 non-null   int64  
 5   medium_completion_rate             3923 non-null   int64  
 6   hard_completion_rate               3923 non-null   float64
 7   session_count                      3923 non-null   int64  
 8   total_session_time                 3923 non-null   int64  
 9   total_ad_viewed                    3923 non-null   int64  
 10  total_earned_reward_amount_in_usd  3923 non-null   float64
dtypes: float64(2), int64(6), object(3)
memory usage: 337.3+ 

In [25]:
# Select relevant features for clustering
features = df[['total_ad_viewed', 'session_count', 'total_session_time', 'total_earned_reward_amount_in_usd']]


for model_config in config['models']:
    model_name = model_config['model_name']
    ModelClass = MODEL_REGISTRY[model_name]


    if model_name == "DBSCANClusterer":
         # Scale the features for DBSCAN (important for distance-based methods)
        scaler = StandardScaler()
        features_scaled = scaler.fit_transform(features)
        for params in model_config['param_grid']:
        
            dbscan = ModelClass(params)
            df['cluster']  = dbscan.fit_predict(features_scaled)
   

# Scale the features for DBSCAN (important for distance-based methods)
#scaler = StandardScaler()
#features_scaled = scaler.fit_transform(features)

# Apply DBSCAN
#dbscan = DBSCAN(eps=0.8, min_samples=5)
#filtered_final_row_data['cluster'] = dbscan.fit_predict(features_scaled)

# Check how many users fall into each cluster
#cluster_counts = filtered_final_row_data['cluster'].value_counts().sort_index()
#cluster_counts


In [26]:
# Calculate average total_ad_viewed per cluster to interpret them
cluster_summary = df.groupby('cluster')['total_ad_viewed'].mean().sort_values()

cluster_summary

cluster
 0     29.791139
 1     31.571429
 4     54.200000
 6    146.250000
 5    171.000000
 3    253.666667
-1    374.777240
 2    452.800000
Name: total_ad_viewed, dtype: float64

Cluster 0 → 🟡 ~43.6 ads viewed → Likely Medium

Cluster 1 → 🔵 ~500.8 ads viewed → Very high → Frequent

Cluster -1 → 🔴 ~552.7 ads viewed → Even higher → Also Frequent

In [27]:
# Map cluster numbers to labels
cluster_label_map = {
    -1: 'Rare',
     0: 'Medium',
     1: 'Frequent'
}

# Apply the mapping to create a new labeled column
df['user_ad_frequency_category'] = df['cluster'].map(cluster_label_map)

# Display a sample of the updated DataFrame
df[['user_key_id', 'total_ad_viewed', 'cluster', 'user_ad_frequency_category']].head()


,user_key_id,total_ad_viewed,cluster,user_ad_frequency_category
0,bVas4SMsXQ,15,0,Medium
1,Hm917zE2Mc,5,0,Medium
2,wFfzWC1Ydt,23,0,Medium
3,73S29MCQWv,3,0,Medium
4,B4A9tTZMAE,1,0,Medium


In [ ]:
from collections import Counter

def select_best_dbscan_config(valid_configs, df_length):
    best_config = None
    best_score = float('-inf')

    for config in valid_configs:
        labels = np.array(config['labels'])

        # Count cluster sizes, exclude noise
        counts = Counter(labels)
        noise_count = counts.pop(-1, 0)
        total_clustered = df_length - noise_count

        if len(counts) != 3:
            continue  # Redundant but safe

        # Score based on cluster balance
        cluster_sizes = np.array(list(counts.values()))
        balance_score = -np.std(cluster_sizes)  # lower std = better balance

        # Final score
        score = total_clustered + balance_score - noise_count

        if score > best_score:
            best_score = score
            best_config = config

    return best_config

# Main model loop
valid_configs = []

for model_config in config['models']:
    model_name = model_config['model_name']
    ModelClass = MODEL_REGISTRY[model_name]

    if model_name == "DBSCANClusterer":
        scaler = StandardScaler()
        features_scaled = scaler.fit_transform(features)

        for params in model_config['param_grid']:
            dbscan = ModelClass(params)
            df['cluster'] = dbscan.fit_predict(features_scaled)

            labels = df['cluster']
            unique_labels = set(labels)
            n_clusters = len(unique_labels - {-1})  # exclude noise

            if n_clusters == 3:
                print(f"✅ Found 3 clusters with: {params}")
                valid_configs.append({
                    "params": params,
                    "labels": labels.tolist()
                })

# Select the best result
best_config = select_best_dbscan_config(valid_configs, len(df))

if best_config:
    print("🎯 Best config with exactly 3 clusters:")
    print(best_config['params'])

    # Optionally reassign and store result
    df['cluster'] = best_config['labels']
else:
    print("❌ No config resulted in exactly 3 clusters.")


✅ Found 3 clusters with: {'eps': 0.5, 'min_samples': 5, 'metric': 'euclidean', 'algorithm': 'auto'}
🎯 Best config with exactly 3 clusters:
{'eps': 0.5, 'min_samples': 5, 'metric': 'euclidean', 'algorithm': 'auto'}
